# 第 5 周练习 — Chiku：Stayez 房地产知识工作者

**学生：** Vagz1216（马丁·卡莫）| **团队：** 欧几里得 | **第 5 周**

---

## 练习目标（理念）

本笔记本把第 5 周的 **RAG（检索增强生成）** 完整管线，接到真实平台 Stayez（stayez.co.ke）上：不是做通用「个人知识库」，而是做一个叫 **Chiku** 的 AI 预订助理，用语义搜索房源 / 体验 / 服务的精选知识库，回答客人问题。

### 业务问题

Stayez 目前把属性数据硬编码在 Python 字典里。平台扩展到 500+ 房源后，复杂查询就很难靠硬编码匹配，例如：

- *「我需要内罗毕一套带泳池的浪漫公寓，价格低于 7,000 肯尼亚先令」*
- *「适合五口之家，带停车位」*
- *「您在火山附近有哪些户外体验？」*

**RAG 怎么解决：** 每个属性是一个 Markdown 文件；新房源 = 新加一个 `.md`。Chiku 用向量语义搜索找到最相关片段，再交给 LLM 生成回答。

### 和第 5 周概念的对应

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 文件加载 | 按文件夹用 `DirectoryLoader` |
| 文本分块 | `RecursiveCharacterTextSplitter`（1000 字符，200 重叠）|
| 嵌入 Embedding | `HuggingFaceEmbeddings(all-MiniLM-L6-v2)`，本地免费 |
| 向量库 | `Chroma` 持久化目录 |
| 可视化 | t-SNE 2D / 3D + Plotly |
| RAG 聊天 | Retriever + Groq / Gemini 注入上下文 |
| 用户界面 | Gradio `ChatInterface` |

## 怎么跑

1. 准备 `.env`：`GROQ_API_KEY`（主）、`GEMINI_API_KEY`（备用）
2. 确保同目录有 `stayez-knowledge-base/`（含 properties / experiences / services）
3. 从上到下运行；最后一格 `demo.launch(inbrowser=True)` 打开 Chiku

In [ ]:
# ========== 导入：环境、数值、绘图、Gradio、LangChain RAG 组件 ==========

# 标准库 os：读环境变量（Environment Variables），例如 API Key
import os
# 标准库 glob：按通配符找文件（本格主要随其它库一起导入）
import glob
# numpy：把嵌入向量转成数组，供 t-SNE 降维
import numpy as np
# plotly.graph_objects：画交互式 2D/3D 散点图
import plotly.graph_objects as go
# gradio：快速搭 Web 聊天界面
import gradio as gr
# Path：用面向对象方式拼知识库路径
from pathlib import Path
# load_dotenv / find_dotenv：定位并加载 .env，避免把密钥写进代码
from dotenv import load_dotenv, find_dotenv
# OpenAI 客户端：这里用兼容接口连 Groq / Gemini
from openai import OpenAI
# TSNE：把高维向量压到 2D/3D 方便人眼观察聚类
from sklearn.manifold import TSNE

# ---------- LangChain：加载 → 分块 → 嵌入 → 向量库 ----------
# DirectoryLoader / TextLoader：按目录批量读 Markdown
from langchain_community.document_loaders import DirectoryLoader, TextLoader
# RecursiveCharacterTextSplitter：按字符递归切块，保留重叠
from langchain_text_splitters import RecursiveCharacterTextSplitter
# HuggingFaceEmbeddings：本地跑小模型，把文本变成向量
from langchain_huggingface import HuggingFaceEmbeddings
# Chroma：持久化向量数据库
from langchain_chroma import Chroma

# 确认本格依赖都已就绪
print("Imports loaded successfully")

In [ ]:
# ========== 配置：加载 API 密钥 + 向量库 / 分块超参 ==========

# find_dotenv 定位 .env；override=True 用文件覆盖进程里已有同名变量
load_dotenv(find_dotenv(), override=True)

# 从环境读取 Groq 密钥（主 LLM 通道）
GROQ_API_KEY   = os.getenv('GROQ_API_KEY')
# 从环境读取 Gemini 密钥（Groq 失败时的备用通道）
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')

# 逐个打印密钥是否已加载（只显示 Found / NOT SET，不打印完整密钥）
for name, key in [("Groq", GROQ_API_KEY), ("Gemini", GEMINI_API_KEY)]:
    print(f"{name}: {'Found' if key else 'NOT SET'}")

# Groq：主 LLM，免费且快；用 OpenAI 兼容 base_url
groq_client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1"
)

# Gemini：备用；同样走 OpenAI 兼容端点（URL 保持原样）
gemini_client = OpenAI(
    api_key=GEMINI_API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

# ---------- 路径与分块 / 嵌入常量（字符串与数值保持原样）----------
# Chroma 持久化目录名
DB_NAME           = "stayez_vector_db"
# 知识库根目录（下含 properties / experiences / services）
KNOWLEDGE_BASE    = Path("stayez-knowledge-base")
# 本地 HuggingFace 嵌入模型 id（首次会下载约 90MB）
EMBEDDING_MODEL   = "all-MiniLM-L6-v2"   # Free HuggingFace model, runs locally
# 每块目标字符数
CHUNK_SIZE        = 1000
# 相邻块重叠字符数，减轻边界处上下文丢失
CHUNK_OVERLAP     = 200

# 打印关键路径与配置，方便核对工作目录
print(f"\nKnowledge base path: {KNOWLEDGE_BASE.resolve()}")
print(f"Vector DB: {DB_NAME}")
print(f"Embedding model: {EMBEDDING_MODEL}")

## A 部分：加载和分块 Stayez 知识库

知识库分成 3 个文件夹，每个文件夹名会写成向量库元数据里的 `doc_type`：

- `properties/` — 单个房源列表（约 7 个属性）
- `experiences/` — 活动与体验
- `services/` — 礼宾与支持服务

下一步用 `DirectoryLoader` 递归读每个文件夹下的 `**/*.md`，再交给分块器。

In [ ]:
# ========== A：按子文件夹加载 Markdown，并打上 doc_type 元数据 ==========

# 汇总所有 Document 对象
documents = []

# 遍历知识库根下的每个子目录（properties / experiences / services）
for folder in KNOWLEDGE_BASE.iterdir():
    # 跳过非目录项（例如散落文件）
    if not folder.is_dir():
        continue
    # 文件夹名即文档类型标签
    doc_type = folder.name
    # DirectoryLoader：对该文件夹递归匹配 **/*.md，用 TextLoader 读 utf-8
    loader = DirectoryLoader(
        str(folder),
        glob="**/*.md",
        loader_cls=TextLoader,
        loader_kwargs={"encoding": "utf-8"}
    )
    # 真正读盘，得到 Document 列表
    folder_docs = loader.load()
    # 给每篇补元数据：类型 + 只保留文件名作 source
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        doc.metadata["source"] = Path(doc.metadata["source"]).name
    # 并入总列表
    documents.extend(folder_docs)
    # 按文件夹汇报加载数量
    print(f"  - Loaded {len(folder_docs)} docs from '{doc_type}/'")

# 总计加载了多少篇
print(f"\nTotal documents loaded: {len(documents)}")

# ---------- 预览第一篇正文与元数据，确认编码 / 标签无误 ----------
print(f"\nFirst document (excerpt):")
print(documents[0].page_content[:300])
print(f"Metadata: {documents[0].metadata}")

In [ ]:
# ========== A：用 RecursiveCharacterTextSplitter 把文档切成块 ==========

# 用前面配置的 CHUNK_SIZE / CHUNK_OVERLAP 建分块器
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)

# 对全部 Document 分块；元数据会随块继承
chunks = splitter.split_documents(documents)

# 打印块数与平均长度，粗看切分是否合理
print(f"Documents split into {len(chunks)} chunks")
print(f"Average chunk length: {sum(len(c.page_content) for c in chunks) // len(chunks)} characters")
# 抽样看第一块正文与元数据
print(f"\nSample chunk:")
print(chunks[0].page_content)
print(f"\nMetadata: {chunks[0].metadata}")

## B 部分：嵌入块并构建 Chroma 向量存储

用 **HuggingFace 的 `all-MiniLM-L6-v2`** 把每个块变成 384 维向量。这个模型：

- 在 CPU 上**本地**跑（嵌入阶段不调云端 API）
- **免费**，无速率限制
- 首次使用自动下载（约 90 MB）
- 对英文文本的语义检索效果足够好

建好后写入 `persist_directory`，下次可直接复用磁盘上的库。

In [ ]:
# ========== B：本地嵌入 + 清空旧库 + Chroma.from_documents 持久化 ==========

# 提示：首次下载嵌入模型可能较慢
print("Loading HuggingFace embedding model (downloads ~90MB on first run)...")
# 按 EMBEDDING_MODEL 名加载本地嵌入函数
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
print("Embedding model ready!")

# 若磁盘上已有同名库：先删集合，保证本次从干净状态重建
if Path(DB_NAME).exists():
    Chroma(persist_directory=DB_NAME, embedding_function=embeddings).delete_collection()
    print(f"Cleared existing vector store'{DB_NAME}'")

# 对所有 chunks 算嵌入并写入 Chroma（persist_directory=DB_NAME）
print("\nBuilding Chroma vector store...")
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=DB_NAME
)

# 取出底层 collection，核对条数与向量维度
collection = vectorstore._collection
count = collection.count()
# 取 1 条嵌入，看维度是否为 384（all-MiniLM-L6-v2）
sample_emb = collection.get(limit=1, include=["embeddings"])["embeddings"][0]

print(f"\nVector store created:")
print(f"  - Total vectors stored: {count:,}")
print(f"  - Embedding dimensions: {len(sample_emb):,} (all-MiniLM-L6-v2)")
print(f"  - Persisted to: ./{DB_NAME}/")

## C 部分：可视化向量空间

用 **t-SNE**（t-分布式随机邻域嵌入）把 384 维压到 2D / 3D。每个点是一块文本，颜色按 `doc_type`（房产 / 体验 / 服务）区分。

> **看图时注意：** RAG 健康时，同类内容会聚在一起；房产块聚一簇、体验块聚一簇。房产簇内部，浪漫 / 豪华房应彼此更近，经济型房也应彼此更近。

In [ ]:
# ========== C：从 collection 取出向量 / 正文 / 元数据，并上色 ==========

# include 指定要取回的字段：嵌入、原文、元数据
result = collection.get(include=["embeddings", "documents", "metadatas"])
# 嵌入矩阵 → numpy，供 TSNE.fit_transform
vectors   = np.array(result["embeddings"])
# 块正文列表（悬停提示用）
docs_text = result["documents"]
# 元数据列表（含 doc_type）
metadatas = result["metadatas"]
# 抽出类型标签序列
doc_types = [m["doc_type"] for m in metadatas]

# 类型 → 颜色；未知类型回退 gray
COLOR_MAP = {"properties": "royalblue", "experiences": "mediumseagreen", "services": "tomato"}
colors = [COLOR_MAP.get(t, "gray") for t in doc_types]

# 汇报向量条数与类型分布
print(f"Data ready for visualization: {len(vectors)} vectors")
from collections import Counter
print(f"Breakdown: {dict(Counter(doc_types))}")

In [ ]:
# ========== C：2D t-SNE + Plotly Scatter ==========

# n_components=2；perplexity 不能 ≥ 样本数，故用 min(5, n-1)
tsne_2d = TSNE(n_components=2, random_state=42, perplexity=min(5, len(vectors)-1))
# 拟合并得到二维坐标
reduced_2d = tsne_2d.fit_transform(vectors)

# 散点：x/y 为降维坐标，颜色按 doc_type，hover 显示类型与正文片段
fig_2d = go.Figure(data=[go.Scatter(
    x=reduced_2d[:, 0],
    y=reduced_2d[:, 1],
    mode="markers",
    marker=dict(size=10, color=colors, opacity=0.85, line=dict(width=1, color="white")),
    text=[f"<b>{t}</b><br>{d[:120]}..." for t, d in zip(doc_types, docs_text)],
    hoverinfo="text"
)])

# 标题与坐标轴（展示用英文标题保持原样）
fig_2d.update_layout(
    title="2D Stayez Knowledge Base — Vector Clusters (t-SNE)",
    xaxis_title="t-SNE Dimension 1",
    yaxis_title="t-SNE Dimension 2",
    width=850, height=600,
    template="plotly_white"
)
# 在笔记本里渲染交互图
fig_2d.show()

In [ ]:
# ========== C：3D t-SNE + Plotly Scatter3d ==========

# 压到 3 维，便于旋转观察簇结构
tsne_3d = TSNE(n_components=3, random_state=42, perplexity=min(5, len(vectors)-1))
reduced_3d = tsne_3d.fit_transform(vectors)

# 三维散点；悬停文本同 2D
fig_3d = go.Figure(data=[go.Scatter3d(
    x=reduced_3d[:, 0],
    y=reduced_3d[:, 1],
    z=reduced_3d[:, 2],
    mode="markers",
    marker=dict(size=6, color=colors, opacity=0.85),
    text=[f"<b>{t}</b><br>{d[:120]}..." for t, d in zip(doc_types, docs_text)],
    hoverinfo="text"
)])

fig_3d.update_layout(
    title="3D Stayez Knowledge Base — Vector Clusters (t-SNE)",
    scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="z"),
    width=900, height=700
)
fig_3d.show()

## D 部分：构建 Chiku — Stayez RAG 预订助手

把 **Retriever** 和 **LLM** 串起来，做成 Chiku。

每条客人消息的流程：

1. 收到问题（例如 *「我需要一间浪漫的公寓」*）
2. **Retriever** 把问题向量化，从 Chroma 取语义最相近的若干块（本实现 `k=3`）
3. 把检索块填进 system prompt 的 `{context}`
4. 优先用 **Groq**，失败再回退 **Gemini**，生成个性化回答

In [ ]:
# ========== D：Retriever + system 模板 + 消息规范化 + Groq/Gemini 回退 ==========

# k=3：少取几块，控制上下文长度，减轻 Groq 免费层 token 压力
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# system 模板：角色设定 + 价格规则 + {context} 占位；正文影响行为，不翻译
CHIKU_SYSTEM = """\
You are Chiku, a warm and knowledgeable guest assistant for Stayez (stayez.co.ke), \
a curated short-stay booking platform based in Kenya.

Your job is to help guests find the perfect property, experience, or service.
You are enthusiastic, friendly, and always recommend booking via stayez.co.ke.
All prices are in Kenyan Shillings (KSh). Always present prices clearly.

Use the context below to answer the guest's question accurately and concisely.
If information is not in the context, say so and suggest the guest visits stayez.co.ke.

RELEVANT STAYEZ CONTEXT:
{context}
"""

def build_messages(message: str, history: list, system_prompt: str) -> list:
    """Build a clean OpenAI-compatible message list.

    Handles any Gradio history format (tuples OR message dicts).
    IMPORTANT: strips any extra fields (e.g. Gradio 5 adds 'metadata')
    since Groq and Gemini only accept 'role' + 'content'.
    Caps history to last 4 messages (2 full turns) to stay within token limits.
    """
    # 规范化后的历史消息列表
    normalised = []
    for item in history:
        # Gradio messages 模式：字典且含 role/content
        if isinstance(item, dict) and "role" in item and "content" in item:
            # 只保留 role + content，丢掉 metadata 等额外字段（Groq/Gemini 更挑剔）
            normalised.append({"role": item["role"], "content": item["content"]})
        # 旧版 (user, bot) 二元组 / 二元列表
        elif isinstance(item, (list, tuple)) and len(item) == 2:
            u, b = item
            if u: normalised.append({"role": "user",      "content": str(u)})
            if b: normalised.append({"role": "assistant", "content": str(b)})
    # 只留最近 4 条消息（约 2 轮），控制 token
    normalised = normalised[-4:]  # keep last 2 turns only
    # system + 历史 + 当前 user
    return [{"role": "system", "content": system_prompt}] + normalised + [{"role": "user", "content": message}]

def chiku_chat(message: str, history: list) -> str:
    # 步骤 1：语义检索，把相关块拼成 context
    docs = retriever.invoke(message)
    context = "\n\n---\n\n".join(doc.page_content for doc in docs)

    # 步骤 2：填模板，并规范化 messages
    system_prompt = CHIKU_SYSTEM.format(context=context)
    messages = build_messages(message, history, system_prompt)

    # 步骤 3：优先 Groq — model id 保持原样
    try:
        response = groq_client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=messages,
            temperature=0.4,
            max_tokens=512
        )
        return response.choices[0].message.content
    except Exception as groq_err:
        # 打印异常类型与信息，便于排查，不中断到下一步回退
        print(f"[Groq error] {type(groq_err).__name__}: {groq_err}")

    # 步骤 4：回退 Gemini — model id 保持原样
    try:
        response = gemini_client.chat.completions.create(
            model="gemini-2.0-flash-lite",
            messages=messages,
            temperature=0.4,
            max_tokens=512
        )
        return response.choices[0].message.content
    except Exception as gemini_err:
        print(f"[Gemini error] {type(gemini_err).__name__}: {gemini_err}")
        # 双通道都失败时的用户可见兜底文案（字符串保持原样）
        return "Chiku is temporarily unavailable. Please visit stayez.co.ke directly."

print("Chiku is ready!")

# ---------- 启动 UI 前自检两轮：验证检索 + 历史规范化 ----------
r1 = chiku_chat("What is your most popular property?", [])
print(f"Turn 1 OK: {r1[:200]}")

hist_test = [{"role": "user", "content": "What is your most popular property?"}, {"role": "assistant", "content": r1}]
r2 = chiku_chat("I need something romantic under KSh 7,000", hist_test)
print(f"Turn 2 OK: {r2[:200]}")

In [ ]:
# ========== D：Gradio Blocks + ChatInterface 启动 Chiku ==========

# Soft 主题；浏览器标题保持原样
with gr.Blocks(theme=gr.themes.Soft(), title="Chiku — Stayez AI Booking Assistant") as demo:
    # 顶部说明 Markdown（文案保持原样，供界面展示）
    gr.Markdown(
        """## Chiku — Your Stayez Booking Assistant
Powered by RAG + Groq. Ask me about properties, experiences, and services at **stayez.co.ke**.

*Try: "I need a romantic apartment under KSh 7,000" or "What outdoor experiences do you have?"*
"""
    )
    # ChatInterface：fn=chiku_chat；type=messages 与上面规范化逻辑对齐
    chat = gr.ChatInterface(
        fn=chiku_chat,
        type="messages",
        # 示例问题：点一下即可试聊（字符串保持原样）
        examples=[
            "What is your most popular property?",
            "I need a romantic place in Nairobi for 2 nights, budget KSh 6,000",
            "Do you have properties for a family of 5?",
            "What outdoor experiences do you offer?",
            "Can you arrange an airport pickup?",
        ],
        # 聊天窗口高度与助手头像 URL（URL 保持原样）
        chatbot=gr.Chatbot(type="messages", height=450, avatar_images=[None, "https://stayez.co.ke/wp-content/uploads/2023/08/stayez-logo.jpg"])
    )

# inbrowser=True：启动后自动打开浏览器
demo.launch(inbrowser=True)

---

## 概括

### 我们建造了什么

适用于 Stayez 的完整 RAG 管道：

1. **Stayez 知识库** — 多个 Markdown（房源 / 体验 / 服务）作可扩展数据源
2. **LangChain 文档加载** — `DirectoryLoader` 按文件夹打上 `doc_type`
3. **`RecursiveCharacterTextSplitter`** — 重叠分块，减轻边界丢上下文
4. **本地嵌入** — `HuggingFaceEmbeddings(all-MiniLM-L6-v2)` → 384 维
5. **Chroma Vector Store** — 持久化到磁盘，可重建也可复用
6. **t-SNE 可视化** — 检查同类块是否聚类
7. **Chiku RAG Chat** — 检索 → 注入 context → Groq（失败则 Gemini）
8. **Gradio UI** — 带示例提示的聊天界面

### 对 Stayez 的价值

团队加新房源时：把新 `.md` 放进 `stayez-knowledge-base/properties/`，重建向量库即可；匹配逻辑由语义检索承担，不必再改硬编码字典。